In [1]:
from sedona.spark import SedonaContext
import os

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/07 18:08:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/07 18:08:18 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/07 18:08:18 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/07 18:08:18 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/07 18:08:18 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/12/07 18:08:18 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/07 18:08:18 WARN SimpleFunctionRegistry: The function st_envelop

# nested loop join

In [3]:
places = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/places")

In [4]:
places.count()

4694140

In [5]:
places.createOrReplaceTempView("places")

In [6]:
sample = places.select("id", "geometry").sample(0.0003)
left = sample.collect()
right = left

In [7]:
def match(left, right):
    return l.geometry.buffer(0.001).intersects(right.geometry) and left.id != right.id

In [8]:
result = []
for l in left:
    for r in right:
        if not match(l, r):
            continue
 
        result.append((l, r))

In [9]:
len(result)

16

# Cartesian Product Join

In [10]:
from shapely.wkt import loads
import pyspark.sql.types as t
import pyspark.sql.functions as f

In [11]:
line = loads("LINESTRING (-46.988525 -23.344778, -46.771545 -23.569022, -46.149445 -23.732555)")

point = loads("POINT(-46.988525 -23.344778)")

points = sedona.createDataFrame([[1, point]])\
    .selectExpr("_1 AS id", "_2 as geom")

lines = sedona.createDataFrame([[1, line]])\
    .selectExpr("_1 AS id", "_2 as geom")

In [12]:
intersects = f.udf(
    lambda left, right: left.intersects(right),
    t.BooleanType()
)
 
result = points\
    .alias("p")\
    .join(
        lines.alias("l"),
        intersects(f.col("p.geom"),
        f.col("l.geom"))
    )

# Spatial Join

In [13]:
# optimized join snippet, using r index

In [14]:
import rtree

In [15]:
r_index = rtree.Rtree()
index = 0
left_mapping = {}

for l in left:
    left_mapping[index] = l
    minx, miny, maxx, maxy = l.geometry.buffer(0.001).bounds

    r_index.add(index, [minx, miny, maxx, maxy])

    index += 1


def get_bounds(geom):
    return geom.geometry.bounds

def predicate(l, r):
    return left_mapping[l].geometry.buffer(0.001).intersects(r.geometry) and left_mapping[l].id != r.id

In [16]:
result_index = []
 
for r in right:
    candidates = r_index.intersection(get_bounds(r))
    candidates_filtered = [[c, r] for c in candidates if predicate(c, r)]
    result_index.extend(candidates_filtered)

In [17]:
len(result_index)

16

# Distributed Spatial Join

In [18]:
# example queries 

## DistanceJoin

```sql
SELECT df1.*
FROM df1, df2
WHERE ST_DistanceSpheroid(df1.geom, df2.geom) < 100
```

```sql
SELECT df1.*
FROM df1
JOIN df2 ON  ST_DistanceSpheroid(df1.geom, df2.geom) < 100
```

## RangeJoin

```sql
SELECT df1.*
FROM df1, df2
WHERE ST_DWithin(df1.geom, df2.geom, 10.0)
```

```sql
SELECT df1.*
FROM df1
JOIN df2 ON ST_DWithin(df1.geom, df2.geom, 10.0)
```

```sql
SELECT left.*
FROM points AS p
JOIN polygons AS pl ON ST_INTERSECTS(p.geom, pl.geom)
```

## Spatial join full dataset

In [19]:
sedona.sql(
"""
    SELECT 
        *
    FROM places AS p1
    JOIN places AS p2 ON ST_DWithin(p1.geometry, p2.geometry, 0.001) AND p1.id <> p2.id
"""
).explain()

== Physical Plan ==
DistanceJoin geometry#25: geometry, geometry#156: geometry, 0.001, true, INTERSECTS, false, ( **org.apache.spark.sql.sedona_sql.expressions.ST_DWithin**   AND NOT (id#24 = id#155))
:- *(1) Filter isnotnull(id#24)
:  +- FileScan geoparquet [id#24,geometry#25,bbox#26,type#27,version#28,sources#29,names#30,categories#31,confidence#32,websites#33,socials#34,emails#35,phones#36,brand#37,addresses#38,geohash#39] Batched: false, DataFilters: [isnotnull(id#24)], Format: GeoParquet, Location: InMemoryFileIndex(1 paths)[s3a://apache-sedona-book/source_data/places], PartitionFilters: [], PushedFilters: [IsNotNull(id)], ReadSchema: struct<id:string,geometry:binary,bbox:struct<xmin:float,xmax:float,ymin:float,ymax:float>,type:st...
+- *(2) Filter isnotnull(id#155)
   +- FileScan geoparquet [id#155,geometry#156,bbox#157,type#158,version#159,sources#160,names#161,categories#162,confidence#163,websites#164,socials#165,emails#166,phones#167,brand#168,addresses#169,geohash#170] Batch

In [20]:
sedona.sql(
"""
    SELECT 
        *
    FROM places AS p1
    JOIN places AS p2 ON ST_DWithin(p1.geometry, p2.geometry, 0.001) AND p1.id <> p2.id
"""
).count()

67300548

In [21]:
places.count()

4694140

# KNN JOIN

In [22]:
properties = [
    # id, lon, lat, value 
    [0, 0, 1, 500_000],
    [1, 1, 8, 400_000],
    [2, 4, 3, 450_000],
    [3, 2, 9, 460_000],
    [4, 10, 12, 480_000],
    [5, 11, 7, 475_000],
    [6, 2, 8, 432_000],
    [7, 5, 3, 456_000],
    [8, 8, 9, 487_000],
    [9, 9, 3, 498_000],
    [10, 3, 2, 434_000],
    [11, 6, 7, 412_000]
]

In [23]:
import math
import statistics

k = 3

def calculate_distance(x1, y1, x2, y2):
    return math.sqrt((x2-x1)**2 + (y2-y1)**2)


result = []
for identifier, lon, lat, value in properties:
    sorted_by_distance = sorted(
        properties,
        key=lambda row: calculate_distance(lon, lat, row[1], row[2])
    )
    
    median_value = statistics.median([
        row[3] for row in sorted_by_distance
    ][1:k+1])

    result.append(median_value)

In [24]:
result

[450000,
 432000,
 456000,
 412000,
 475000,
 487000,
 412000,
 450000,
 475000,
 456000,
 456000,
 456000]

# Distributed KNN JOIN with Sedona

In [25]:
knn_result = sedona.sql(
    """
    SELECT p1.*, p2.id AS p2_id, p2.geometry AS p2_geometry
    FROM places AS p1
    JOIN places AS p2 ON ST_KNN(p1.geometry, p2.geometry, 4)
    """
)

In [26]:
knn_result.where("id <> p2_id").show()

[Stage 26:>                                                         (0 + 1) / 1]

+--------------------+--------------------+--------------------+-----+-------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+------+---------------+-----+--------------------+-------+--------------------+--------------------+
|                  id|            geometry|                bbox| type|version|             sources|               names|          categories|        confidence|            websites|             socials|emails|         phones|brand|           addresses|geohash|               p2_id|         p2_geometry|
+--------------------+--------------------+--------------------+-----+-------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+------+---------------+-----+--------------------+-------+--------------------+--------------------+
|08f1e35044b60a9b0...|POINT (15.1282226...|{15.128222, 15.12...|place|      0|[{, meta, 846

In [27]:
places.count()

4694140